# Chapter 7 — Enterprise RAG: Agentic Routing, Semantic Caching, and Query Rewriting

Companion code for **Chapter 7** of *Build an Advanced RAG Application (From Scratch)*.

The plain RAG of Chapter 6 starts to crack in enterprise settings — multiple knowledge bases, paraphrased questions hitting the LLM 100 times an hour, vague or compound queries that no single search nails. Chapter 7 fixes all three with three pillars:

| Pillar | What it does | Module |
|--------|-------------|--------|
| **Agentic Router** | Classifies each query and dispatches to the right knowledge base (or web) | [agentic_router.py](agentic_router.py) |
| **Semantic Cache** | Reuses prior answers for paraphrases of the same question | [semantic_cache.py](semantic_cache.py) |
| **Query Rewriter / Decomposer** | Polishes vague queries; splits compound ones into atomic sub-queries | [query_rewriter.py](query_rewriter.py) |
| **Combined pipeline** | All three plus a time-sensitivity bypass | [enterprise_pipeline.py](enterprise_pipeline.py) |

The data — **OpenAI's "Practical Guide to Building Agents"** and **Uber/Lyft 10-K filings** — lives in [`data/`](data/) and is ingested via [`ingest.py`](ingest.py) into two Qdrant collections (`opnai_data`, `10k_data`).

## 0. Setup

Required keys in your `.env` (at the repo root):

| Key | Required? | Used for |
|-----|-----------|----------|
| `OPENAI_API_KEY` | yes | LLM calls (router, rewriter, generator) |
| `QDRANT_URL` | optional | Remote Qdrant. Falls back to in-memory for local demo |
| `QDRANT_API_KEY` | optional | Cloud Qdrant auth |
| `SERPAPI_KEY` | optional | Real web search. Without it, `search_web` returns a stub |

Embeddings use **`nomic-embed-text-v1.5`** locally (768d) — same encoder the original course used.

In [ ]:
import sys, os, asyncio, time
sys.path.insert(0, '.')

import nest_asyncio
nest_asyncio.apply()  # so we can `await` inside Jupyter

from agentic_router import (
    get_qdrant_async_client,
    route_query,
    handle_query,
    rag_formatted_response,
    search_web,
)
from semantic_cache import SemanticCaching, is_time_sensitive
from query_rewriter import rewrite_query, decompose_query
from enterprise_pipeline import enterprise_rag_pipeline
from ingest import ingest_all

## 1. Ingest the corpora into Qdrant

We use a single in-memory Qdrant client for the whole notebook. The first run ingests:

- 1 OpenAI agents PDF -> `opnai_data` collection
- 3 Lyft 10-Ks (2020-2022) + 1 Uber 10-K 2021 -> `10k_data` collection

Embedding 4-5 PDFs takes a couple of minutes on CPU and finishes much faster on GPU/MPS. Set `QDRANT_URL` to a persistent cluster if you don't want to re-ingest each session.

In [ ]:
qdrant = get_qdrant_async_client()
asyncio.run(ingest_all(qdrant))

## 2. Pillar 1 — Agentic Routing

The router is a **single GPT-4o call** that returns a JSON envelope:

```json
{ "action": "OPENAI_QUERY|10K_DOCUMENT_QUERY|WEB_SEARCH", "reason": "...", "answer": "..." }
```

If the question is trivial (e.g. "What does SDK stand for?"), it answers directly in 5 words and skips retrieval entirely.

In [ ]:
for q in [
    "How do I create an OpenAI assistant with file search?",
    "What was Uber's gross bookings in Q3 2021?",
    "What are the most popular open-source LLMs in 2025?",
    "What does SDK stand for?",
]:
    print(f"Q: {q}")
    decision = route_query(q)
    print(f"  -> {decision['action']}  ({decision['reason']})")
    if decision['answer']: print(f"     direct answer: {decision['answer']}")
    print()

### End-to-end via `handle_query`

This runs the route -> retrieve -> respond loop without caching or rewriting yet — the simplest agentic-router shape.

In [ ]:
print(asyncio.run(handle_query(
    "How do I create an OpenAI assistant with file search?", qdrant
)))

In [ ]:
print(asyncio.run(handle_query(
    "What was Uber's revenue in 2021?", qdrant
)))

## 3. Pillar 2 — Semantic Cache

The cache stores `(question, embedding, answer)` triples. On a new question we embed it and FAISS-search for the nearest cached embedding. Below the L2 threshold of `0.2` we return the cached answer — no LLM call, no retrieval.

Time-sensitive queries ("today", "latest", "stock price"…) are detected upfront and bypass the cache entirely.

In [ ]:
# Time-sensitivity gate
for q in ["What is OpenAI?", "What's the weather today?", "Latest GPT-5 news"]:
    print(f"  {is_time_sensitive(q):>5}  {q}")

In [ ]:
# Build a fresh cache (clear_on_init=True wipes any prior cache.json)
cache = SemanticCaching(clear_on_init=True)

q1 = "What was Uber's revenue in 2021?"
hit, _, embedding, _, _ = cache.check_cache(q1)
assert not hit
answer = asyncio.run(handle_query(q1, qdrant))
cache.add_to_cache(q1, answer, embedding)
print('First call (miss):\n', answer[:240], '...\n')

# Paraphrase
q2 = "How much did Uber earn in fiscal year 2021?"
hit, cached, _, sim, _ = cache.check_cache(q2)
print(f'Paraphrase hit? {hit}  (sim={sim:.3f})')
print('Cached answer:\n', cached[:240], '...')

## 4. Pillar 3 — Query Rewriting and Decomposition

Two LLM calls, both with `temperature=0`:

- **`rewrite_query`** expands abbreviations ("Q3" -> "third quarter"), fills in vague references using the last 3 conversation turns, and adds clearly-implied context. It does NOT invent constraints.
- **`decompose_query`** splits a compound question into 2-4 atomic sub-queries. Returns the original unchanged when it isn't compound.

In [ ]:
# Rewriting with conversation history
history = [
    ("Tell me about Uber's 2021 financials",
     "Uber's 2021 revenue was $17.5B, up 57% YoY..."),
]
print(rewrite_query("How does that compare to Lyft?", conversation_history=history))

In [ ]:
# Decomposition
compound = "What was Uber's revenue in 2021 and how does their gross bookings growth compare to Lyft's?"
for sq in decompose_query(compound):
    print(' -', sq)

## 5. Combined: the Enterprise RAG pipeline

`enterprise_rag_pipeline` runs all three pillars in order:

1. Cache check (paraphrase-aware)
2. Time-sensitivity bypass
3. Routing
4. Rewriting
5. Decomposition
6. Per-sub-query retrieval
7. Grounded synthesis with citations
8. Cache store

It returns a dict with the full audit trail — so you can log *why* the system answered the way it did.

In [ ]:
cache = SemanticCaching(clear_on_init=True)

test_queries = [
    ('Cache miss, 10-K route',        "What was Uber's revenue in 2021?"),
    ('Cache hit on paraphrase',       'How much money did Uber make in 2021?'),
    ('Time-sensitive bypass',         'What are the latest AI model releases this week?'),
    ('Compound, needs decomposition', 'Q3 rev breakdown for the two ride-share companies'),
    ('OpenAI docs route',             'How do I create an assistant with file search?'),
]

for label, q in test_queries:
    print('=' * 70)
    print(f'[{label}]  {q}')
    print('=' * 70)
    result = asyncio.run(enterprise_rag_pipeline(q, cache, qdrant))
    print(f"cache_hit       : {result['cache_hit']}")
    print(f"time_sensitive  : {result['time_sensitive']}")
    print(f"route           : {result['route']}")
    print(f"reason          : {result['reason']}")
    print(f"rewritten_query : {result['rewritten_query']}")
    print(f"sub_queries     : {result['sub_queries']}")
    print(f"answer          : {(result['answer'] or '')[:240]}...")
    print()

## 6. Cache speedup, measured

The whole point of the cache is latency. Run the same paraphrase twice and observe.

In [ ]:
cache = SemanticCaching(clear_on_init=True)

t0 = time.time()
asyncio.run(enterprise_rag_pipeline("What was Uber's revenue in 2021?", cache, qdrant))
miss_time = time.time() - t0

t0 = time.time()
asyncio.run(enterprise_rag_pipeline('How much did Uber earn in fiscal year 2021?', cache, qdrant))
hit_time = time.time() - t0

print(f'Cache MISS: {miss_time:.2f}s')
print(f'Cache HIT:  {hit_time:.3f}s')
print(f'Speedup:    {miss_time / max(hit_time, 1e-3):.0f}x')

## What's next

Chapter 8 takes the pieces from Chapters 6 and 7 — retrieval, prompting, routing, caching, rewriting — and packages them for production: deployment, observability, guardrails, and full agentic orchestration where autonomous agents drive the pipeline.